# M4-L2 — Classical nucleation theory: the assembly phase diagram

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in `M4_theory_compendium.pdf`, a separate document that is not included in this repository.

**What this layer does.** L1 answers *what happens at one concentration*; L2 answers *where in parameter space assembly is possible at all*. A partial shell of $n$ subunits pays an edge tension on its unsatisfied rim and gains a bulk contact energy, which gives a droplet free energy $G(n) = -n\,\Delta\mu + \sigma a_1 n^{2/3}$ with a barrier at $n^\* = (2\sigma a_1/3\Delta\mu)^3$.

Three regimes follow: no drive (no assembly), a barrier of a few $kT$ (productive assembly), and a vanished barrier (many simultaneous nuclei, that is, a kinetic trap). The trap that L1 finds numerically appears here as a region of parameter space and not as a single point.

**Runtime:** seconds.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile brusafe_mod4_nucleation.py
"""
BruSafe Module 4 (WIDENED) — Assembly Thermodynamics: Nucleation Phase Diagram
==============================================================================
The original module used a nucleation-elongation cascade with a fixed critical
nucleus. We widen to classical nucleation theory (CNT) for capsid assembly and
map the assembly PHASE DIAGRAM in (free-subunit concentration, binding-energy)
space, the central object in the virus-assembly literature (Endres & Zlotnick
2002; Hagan 2014).

CNT free energy of a partial shell of n subunits (droplet form):
  G(n) = -n * dmu  +  sigma * a1 * n^{2/3}
where
  dmu  = -g_b * cbar  +  kT ln(C/C0)   is the per-subunit thermodynamic drive
         (g_b<0 favourable per-contact energy, cbar closed-shell contacts/subunit;
         the ln term is the translational-entropy cost of recruiting a subunit),
  sigma = line/edge tension of an incomplete shell (unsatisfied rim contacts),
  a1    = geometric prefactor.

For dmu>0 (supersaturation) there is a nucleation barrier at
  n* = (2 sigma a1 / (3 dmu))^3 ,   G* = (4/27) (sigma a1)^3 / dmu^2 .
Regimes: dmu<=0 -> NO assembly; moderate dmu with G*>~a few kT -> PRODUCTIVE
(nucleation-limited, monomer not exhausted); very large dmu (strong g_b AND high
C) -> G*->0, many simultaneous nuclei -> KINETIC TRAP (the non-monotonic yield
the coarse-grained MC and the cascade model both reproduced).

Refs: Endres & Zlotnick 2002 (10.1016/s0006-3495(02)75245-4);
Hagan 2014 (10.1002/9781118755815.ch01); Zandi et al. 2004 (10.1073/pnas.0405844101).
"""
import numpy as np

NCAP = 90
RT   = 0.593           # kcal/mol at 298 K
kT   = RT
CBAR = 1.0             # net contacts per dimer entering the drive (calibrated)
SIGMA= 1.6             # edge tension (kcal/mol) of incomplete-rim contacts
A1   = 1.5             # geometric prefactor
C0   = 500.0           # reference concentration (uM), standard-state scale

def dmu(g_b, C):
    return -g_b*CBAR - kT*np.log(C0/max(C,1e-9))   # kcal/mol per subunit; C<C0 penalised

def nstar_Gstar(g_b, C):
    dm = dmu(g_b,C)
    if dm<=0:
        return np.inf, np.inf, dm     # no supersaturation -> no assembly
    nstar = (2*SIGMA*A1/(3*dm))**3
    Gstar = (4/27)*(SIGMA*A1)**3/dm**2
    return nstar, Gstar, dm

def classify(g_b, C):
    nstar,Gstar,dm = nstar_Gstar(g_b,C)
    if dm<=0:            return 0, nstar, Gstar   # no-assembly
    if Gstar < 1.0:      return 2, nstar, Gstar   # barrier gone -> kinetic trap
    if Gstar > 35:       return 0, nstar, Gstar   # barrier too high on timescale
    return 1, nstar, Gstar                        # productive

def G_curve(g_b, C):
    dm=dmu(g_b,C); ns=np.arange(1,NCAP+1)
    return ns, -ns*dm + SIGMA*A1*ns**(2/3)

def run():
    g_bs=np.linspace(-1.0,-5.0,60)
    Cs  =np.logspace(-0.5,2.5,60)
    phase=np.zeros((len(g_bs),len(Cs))); nstar_map=np.zeros_like(phase); Gstar_map=np.zeros_like(phase)
    for i,g in enumerate(g_bs):
        for j,C in enumerate(Cs):
            ph,ns,Gs=classify(g,C); phase[i,j]=ph
            nstar_map[i,j]=min(ns,NCAP) if np.isfinite(ns) else NCAP
            Gstar_map[i,j]=min(Gs,50) if np.isfinite(Gs) else 50
    ns,Gc=G_curve(-1.5,120.0)
    nstar,Gstar,dm=nstar_Gstar(-1.5,120.0)
    return dict(g_bs=g_bs,Cs=Cs,phase=phase,nstar_map=nstar_map,Gstar_map=Gstar_map,
                example=dict(ns=ns,Gc=Gc,nstar=nstar,Gstar=Gstar,dm=dm))

if __name__=="__main__":
    R=run(); ph=R["phase"]
    print("=== Assembly phase diagram (concentration x binding energy) ===")
    print(f"  no-assembly fraction  = {(ph==0).mean():.2f}")
    print(f"  productive  fraction  = {(ph==1).mean():.2f}")
    print(f"  kinetic-trap fraction = {(ph==2).mean():.2f}")
    ex=R["example"]
    print(f"\n=== Example barrier (g_b=-1.5 kcal/mol/contact, C=120 uM) ===")
    print(f"  per-subunit drive dmu = {ex['dm']:.2f} kcal/mol")
    print(f"  critical nucleus n*   = {ex['nstar']:.1f} dimers")
    print(f"  nucleation barrier G* = {ex['Gstar']:.2f} kcal/mol = {ex['Gstar']/RT:.1f} kT")


## 1. Barrier and critical nucleus at one operating point

In [ ]:
import brusafe_mod4_nucleation as L2

g_b, C = -1.5, 120.0   # kcal/mol per contact, uM
nstar, Gstar, dm = L2.nstar_Gstar(g_b, C)
print(f'per-subunit drive  dmu = {dm:.2f} kcal/mol')
print(f'critical nucleus    n* = {nstar:.1f} dimers')
print(f'barrier             G* = {Gstar:.2f} kcal/mol = {Gstar/L2.RT:.1f} kT')

ns, Gc = L2.G_curve(g_b, C)
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.plot(ns, Gc, lw=2, color='#0b6e5a')
ax.axhline(0, color='0.7', lw=0.8)
ax.plot([nstar], [Gstar], 'o', color='#b3402f', ms=7, zorder=5)
ax.annotate(f'$n^*$={nstar:.0f}, $G^*$={Gstar:.1f} kcal/mol',
            xy=(nstar, Gstar), xytext=(nstar+12, Gstar*0.8), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='0.4'))
ax.set_xlabel('partial-shell size $n$ (dimers)'); ax.set_ylabel('$G(n)$ (kcal/mol)')
ax.set_title('Droplet free energy of a growing shell')
fig.tight_layout(); fig.savefig('M4_L2_barrier.png', dpi=200)

## 2. The phase diagram

60×60 sweep over per-contact energy and free-subunit concentration. Boundaries are the ones coded in `classify`: no drive or barrier above 35 kcal/mol → no assembly; barrier below 1 kcal/mol → kinetic trap; otherwise productive.

In [ ]:
R = L2.run(); ph = R['phase']
print(f'no-assembly   fraction of grid = {(ph==0).mean():.2f}')
print(f'productive    fraction of grid = {(ph==1).mean():.2f}')
print(f'kinetic-trap  fraction of grid = {(ph==2).mean():.2f}')

from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#e6e3dc', '#0b6e5a', '#b3402f'])
fig, ax = plt.subplots(figsize=(6.2, 4.2))
im = ax.pcolormesh(R['Cs'], R['g_bs'], ph, cmap=cmap, vmin=-0.5, vmax=2.5, shading='auto')
ax.set_xscale('log')
ax.set_xlabel(r'free subunit concentration $C$ ($\mu$M)')
ax.set_ylabel(r'per-contact energy $g_b$ (kcal/mol)')
ax.set_title('M4-L2 assembly phase diagram')
cb = fig.colorbar(im, ax=ax, ticks=[0, 1, 2])
cb.ax.set_yticklabels(['no assembly', 'productive', 'kinetic trap'])
fig.tight_layout(); fig.savefig('M4_L2_phase_diagram.png', dpi=200)

## 3. Locating a bench condition on the phase diagram

The two numbers in the cell below are the measured free-dimer concentration and the fitted per-contact energy (from `M4_FIT_to_bench_data.ipynb`). The cell reports the regime in which the condition falls and the direction in which it should be moved.

In [ ]:
C_bench, g_bench = 2.5, -4.0   # measured [D] (uM) and fitted g_contact (kcal/mol); example values
reg, nstar, Gstar = L2.classify(g_bench, C_bench)
label = ['NO ASSEMBLY', 'PRODUCTIVE', 'KINETIC TRAP'][reg]
print(f'regime: {label}   (n* = {nstar:.1f} dimers, G* = {Gstar:.2f} kcal/mol)')
if reg == 2:
    print('-> lower the concentration or weaken the interface (less salt / higher pH)')
elif reg == 0:
    print('-> raise the concentration or strengthen the interface (more salt / lower pH)')
else:
    print('-> you are in the productive window; optimise yield with the L1 sweep')

---
**Outputs:** `M4_L2_barrier.png`, `M4_L2_phase_diagram.png`.